# 12. Instacart-SelF 카테고리 매핑

> **목표**: Instacart aisle → SelF category 매핑 테이블 생성

## 매핑 방법
1. 이름 유사도 기반 자동 매핑 (Fuzzy Matching)
2. 수동 검토 및 보정
3. 매핑 불가 aisle 처리 방안

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from difflib import SequenceMatcher
import warnings

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/instacart')
PROCESSED_DIR = Path('../data/processed/instacart')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Instacart Aisle 데이터 로드

In [2]:
# Instacart 데이터 로드
aisles = pd.read_csv(DATA_DIR / 'aisles.csv')
departments = pd.read_csv(DATA_DIR / 'departments.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')

# 상품에 aisle, department 정보 병합
products_full = products.merge(aisles, on='aisle_id').merge(departments, on='department_id')

# Aisle 목록 (department 포함)
aisle_list = products_full[['aisle_id', 'aisle', 'department_id', 'department']].drop_duplicates()
aisle_list = aisle_list.sort_values('aisle_id').reset_index(drop=True)

print(f"Instacart Aisle 수: {len(aisle_list)}")
print(f"Department 수: {len(departments)}")

Instacart Aisle 수: 134
Department 수: 21


In [3]:
# Aisle 전체 목록 출력
print("=" * 80)
print("Instacart Aisle 전체 목록 (134개)")
print("=" * 80)
for _, row in aisle_list.iterrows():
    print(f"[{row['aisle_id']:3d}] {row['aisle']:<45} ({row['department']})")

Instacart Aisle 전체 목록 (134개)
[  1] prepared soups salads                         (deli)
[  2] specialty cheeses                             (dairy eggs)
[  3] energy granola bars                           (snacks)
[  4] instant foods                                 (dry goods pasta)
[  5] marinades meat preparation                    (pantry)
[  6] other                                         (other)
[  7] packaged meat                                 (meat seafood)
[  8] bakery desserts                               (bakery)
[  9] pasta sauce                                   (dry goods pasta)
[ 10] kitchen supplies                              (household)
[ 11] cold flu allergy                              (personal care)
[ 12] fresh pasta                                   (dry goods pasta)
[ 13] prepared meals                                (deli)
[ 14] tofu meat alternatives                        (deli)
[ 15] packaged seafood                              (meat seafood)
[ 16] fres

## 2. SelF 카테고리 구조 정의

SelF 서비스의 카테고리 구조를 정의합니다. 실제 DB의 categories 테이블 구조에 맞게 수정 필요.

In [4]:
# SelF 카테고리 구조 (예시 - 실제 DB 구조에 맞게 수정 필요)
# 이 데이터는 실제 SelF DB의 categories 테이블에서 가져와야 합니다

self_categories = {
    # 신선식품
    1: {'name': '과일', 'parent': None, 'keywords': ['fruit', 'fresh fruit', 'apple', 'banana', 'orange']},
    2: {'name': '채소', 'parent': None, 'keywords': ['vegetable', 'fresh vegetable', 'lettuce', 'tomato', 'onion']},
    3: {'name': '샐러드/간편채소', 'parent': None, 'keywords': ['salad', 'packaged', 'prepared']},
    
    # 육류/해산물
    10: {'name': '소고기', 'parent': None, 'keywords': ['beef', 'steak']},
    11: {'name': '돼지고기', 'parent': None, 'keywords': ['pork', 'bacon', 'ham']},
    12: {'name': '닭고기', 'parent': None, 'keywords': ['chicken', 'poultry', 'turkey']},
    13: {'name': '해산물', 'parent': None, 'keywords': ['seafood', 'fish', 'shrimp', 'salmon']},
    
    # 유제품
    20: {'name': '우유', 'parent': None, 'keywords': ['milk', 'dairy']},
    21: {'name': '요거트', 'parent': None, 'keywords': ['yogurt', 'greek']},
    22: {'name': '치즈', 'parent': None, 'keywords': ['cheese', 'cream cheese']},
    23: {'name': '달걀', 'parent': None, 'keywords': ['egg', 'eggs']},
    24: {'name': '버터/크림', 'parent': None, 'keywords': ['butter', 'cream', 'margarine']},
    
    # 냉동식품
    30: {'name': '냉동식품', 'parent': None, 'keywords': ['frozen', 'frozen meal', 'frozen dinner']},
    31: {'name': '아이스크림', 'parent': None, 'keywords': ['ice cream', 'frozen dessert', 'gelato']},
    32: {'name': '냉동채소', 'parent': None, 'keywords': ['frozen vegetable', 'frozen fruit']},
    33: {'name': '냉동피자', 'parent': None, 'keywords': ['frozen pizza', 'pizza']},
    
    # 음료
    40: {'name': '생수', 'parent': None, 'keywords': ['water', 'sparkling', 'seltzer']},
    41: {'name': '탄산음료', 'parent': None, 'keywords': ['soda', 'soft drink', 'cola']},
    42: {'name': '주스', 'parent': None, 'keywords': ['juice', 'nectar', 'orange juice']},
    43: {'name': '커피', 'parent': None, 'keywords': ['coffee', 'espresso', 'instant coffee']},
    44: {'name': '차', 'parent': None, 'keywords': ['tea', 'green tea', 'herbal']},
    
    # 과자/간식
    50: {'name': '과자', 'parent': None, 'keywords': ['chip', 'pretzel', 'cracker', 'snack']},
    51: {'name': '쿠키/케이크', 'parent': None, 'keywords': ['cookie', 'cake', 'brownie', 'pastry']},
    52: {'name': '초콜릿/캔디', 'parent': None, 'keywords': ['chocolate', 'candy', 'gum', 'mint']},
    53: {'name': '견과류', 'parent': None, 'keywords': ['nut', 'seed', 'dried fruit', 'almond']},
    
    # 베이커리
    60: {'name': '빵', 'parent': None, 'keywords': ['bread', 'bun', 'roll', 'bagel', 'tortilla']},
    61: {'name': '케이크/디저트', 'parent': None, 'keywords': ['bakery dessert', 'pie', 'tart']},
    
    # 양념/소스
    70: {'name': '소스', 'parent': None, 'keywords': ['sauce', 'pasta sauce', 'marinara', 'salsa']},
    71: {'name': '양념', 'parent': None, 'keywords': ['condiment', 'ketchup', 'mustard', 'mayonnaise']},
    72: {'name': '식용유', 'parent': None, 'keywords': ['oil', 'olive oil', 'cooking oil', 'vinegar']},
    73: {'name': '향신료', 'parent': None, 'keywords': ['spice', 'seasoning', 'herb', 'salt', 'pepper']},
    
    # 식료품
    80: {'name': '면/파스타', 'parent': None, 'keywords': ['pasta', 'noodle', 'spaghetti', 'ramen']},
    81: {'name': '쌀/곡물', 'parent': None, 'keywords': ['rice', 'grain', 'quinoa', 'oat']},
    82: {'name': '통조림', 'parent': None, 'keywords': ['canned', 'can', 'soup', 'bean']},
    83: {'name': '시리얼', 'parent': None, 'keywords': ['cereal', 'granola', 'oatmeal', 'breakfast']},
    
    # 간편식
    90: {'name': '간편식/도시락', 'parent': None, 'keywords': ['prepared', 'ready', 'meal', 'lunch']},
    91: {'name': '델리/샐러드', 'parent': None, 'keywords': ['deli', 'salad', 'prepared salad']},
    
    # 주류
    100: {'name': '맥주', 'parent': None, 'keywords': ['beer', 'ale', 'lager']},
    101: {'name': '와인', 'parent': None, 'keywords': ['wine', 'red wine', 'white wine']},
    102: {'name': '양주/기타주류', 'parent': None, 'keywords': ['spirit', 'liquor', 'whiskey', 'vodka']},
}

print(f"SelF 카테고리 수: {len(self_categories)}")

SelF 카테고리 수: 40


## 3. 자동 매핑 (키워드 기반)

In [5]:
def find_best_match(aisle_name, self_categories):
    """Instacart aisle에 가장 적합한 SelF 카테고리 찾기"""
    aisle_lower = aisle_name.lower()
    best_match = None
    best_score = 0
    
    for cat_id, cat_info in self_categories.items():
        # 키워드 매칭
        for keyword in cat_info['keywords']:
            keyword_lower = keyword.lower()
            
            # 정확 매칭
            if keyword_lower in aisle_lower or aisle_lower in keyword_lower:
                score = len(keyword_lower) / len(aisle_lower) if len(aisle_lower) > 0 else 0
                if keyword_lower == aisle_lower:
                    score = 1.0
                if score > best_score:
                    best_score = score
                    best_match = cat_id
            
            # 부분 매칭 (SequenceMatcher)
            ratio = SequenceMatcher(None, aisle_lower, keyword_lower).ratio()
            if ratio > 0.6 and ratio > best_score:
                best_score = ratio
                best_match = cat_id
    
    return best_match, best_score

# 자동 매핑 실행
auto_mapping = []
for _, row in aisle_list.iterrows():
    match_id, score = find_best_match(row['aisle'], self_categories)
    auto_mapping.append({
        'aisle_id': row['aisle_id'],
        'aisle': row['aisle'],
        'department': row['department'],
        'self_category_id': match_id,
        'self_category_name': self_categories[match_id]['name'] if match_id else None,
        'match_score': score,
    })

auto_mapping_df = pd.DataFrame(auto_mapping)
print(f"자동 매핑 결과: {len(auto_mapping_df)}개")
print(f"매핑 성공: {auto_mapping_df['self_category_id'].notna().sum()}개")
print(f"매핑 실패: {auto_mapping_df['self_category_id'].isna().sum()}개")

자동 매핑 결과: 134개
매핑 성공: 87개
매핑 실패: 47개


In [6]:
# 높은 신뢰도 매핑 (score >= 0.5)
high_confidence = auto_mapping_df[auto_mapping_df['match_score'] >= 0.5]
print(f"\n높은 신뢰도 매핑 (score >= 0.5): {len(high_confidence)}개")
display(high_confidence.head(20))


높은 신뢰도 매핑 (score >= 0.5): 64개


,aisle_id,aisle,department,self_category_id,self_category_name,match_score
0,1,prepared soups salads,deli,91.0,델리/샐러드,0.800000
1,2,specialty cheeses,dairy eggs,22.0,치즈,0.620690
3,4,instant foods,dry goods pasta,43.0,커피,0.666667
5,6,other,other,73.0,향신료,0.666667
6,7,packaged meat,meat seafood,3.0,샐러드/간편채소,0.761905
7,8,bakery desserts,bakery,61.0,케이크/디저트,0.965517
8,9,pasta sauce,dry goods pasta,70.0,소스,1.000000
11,12,fresh pasta,dry goods pasta,1.0,과일,0.636364
12,13,prepared meals,deli,91.0,델리/샐러드,0.785714
14,15,packaged seafood,meat seafood,3.0,샐러드/간편채소,0.666667


In [7]:
# 낮은 신뢰도 또는 매핑 실패 (수동 검토 필요)
low_confidence = auto_mapping_df[
    (auto_mapping_df['match_score'] < 0.5) | 
    (auto_mapping_df['self_category_id'].isna())
]
print(f"\n수동 검토 필요: {len(low_confidence)}개")
display(low_confidence)


수동 검토 필요: 70개


,aisle_id,aisle,department,self_category_id,self_category_name,match_score
2,3,energy granola bars,snacks,83.0,시리얼,0.368421
4,5,marinades meat preparation,pantry,NaN,None,0.000000
9,10,kitchen supplies,household,NaN,None,0.000000
10,11,cold flu allergy,personal care,NaN,None,0.000000
13,14,tofu meat alternatives,deli,NaN,None,0.000000
...,...,...,...,...,...,...
128,129,frozen appetizers sides,frozen,30.0,냉동식품,0.260870
129,130,hot cereal pancake mixes,breakfast,83.0,시리얼,0.250000
131,132,beauty,personal care,NaN,None,0.000000
132,133,muscles joints pain relief,personal care,NaN,None,0.000000


## 4. 수동 매핑 보정

In [8]:
# 수동 매핑 (자동 매핑 실패 또는 보정이 필요한 경우)
# 실제 SelF 카테고리 ID에 맞게 수정 필요

manual_mapping = {
    # 신선식품
    24: 1,   # fresh fruits → 과일
    123: 1,  # packaged produce → 과일 (또는 채소)
    83: 2,   # fresh vegetables → 채소
    3: 3,    # specialty cheeses → 치즈 (보정)
    
    # 유제품
    84: 20,  # milk → 우유
    121: 21, # yogurt → 요거트
    21: 22,  # packaged cheese → 치즈
    86: 23,  # eggs → 달걀
    91: 24,  # butter → 버터/크림
    
    # 육류/해산물
    35: 12,  # poultry counter → 닭고기
    106: 12, # packaged poultry → 닭고기
    1: 10,   # prepared soups salads (보정) → 간편식
    
    # 냉동식품
    37: 30,  # frozen breakfast → 냉동식품
    38: 30,  # frozen appetizers sides → 냉동식품
    39: 31,  # ice cream toppings → 아이스크림
    
    # 음료
    115: 40, # water seltzer sparkling water → 생수
    77: 41,  # soft drinks → 탄산음료
    98: 43,  # coffee → 커피
    94: 44,  # tea → 차
    
    # 과자/간식
    107: 50, # chips pretzels → 과자
    45: 51,  # cookies cakes → 쿠키/케이크
    119: 52, # candy chocolate → 초콜릿/캔디
    117: 53, # nuts seeds dried fruit → 견과류
    
    # 베이커리
    112: 60, # bread → 빵
    128: 60, # bakery desserts → 케이크/디저트
    
    # 양념/소스
    9: 70,   # pasta sauce → 소스
    5: 71,   # spreads → 양념
    13: 71,  # pickled goods olives → 양념
    
    # 식료품
    129: 80, # dry pasta → 면/파스타
    130: 82, # canned meals beans → 통조림
    131: 83, # cereal → 시리얼
    
    # 간편식
    20: 90,  # lunch meat → 간편식
    
    # 비식품 (매핑 제외)
    11: None,  # cleaning products
    55: None,  # first aid
    71: None,  # baby accessories
    72: None,  # baby bath body care
    73: None,  # baby food formula
    74: None,  # diapers wipes
    126: None, # dog food care
    127: None, # cat food care
}

print(f"수동 매핑 정의: {len(manual_mapping)}개")

수동 매핑 정의: 40개


In [9]:
# 최종 매핑 테이블 생성 (자동 + 수동 병합)
final_mapping = auto_mapping_df.copy()

# 수동 매핑으로 덮어쓰기
for aisle_id, self_cat_id in manual_mapping.items():
    mask = final_mapping['aisle_id'] == aisle_id
    if mask.any():
        final_mapping.loc[mask, 'self_category_id'] = self_cat_id
        if self_cat_id and self_cat_id in self_categories:
            final_mapping.loc[mask, 'self_category_name'] = self_categories[self_cat_id]['name']
        else:
            final_mapping.loc[mask, 'self_category_name'] = None
        final_mapping.loc[mask, 'match_score'] = 1.0  # 수동 매핑은 최고 점수

# 매핑 통계
mapped_count = final_mapping['self_category_id'].notna().sum()
unmapped_count = final_mapping['self_category_id'].isna().sum()

print(f"\n=== 최종 매핑 결과 ===")
print(f"총 Aisle: {len(final_mapping)}개")
print(f"매핑 완료: {mapped_count}개 ({mapped_count/len(final_mapping)*100:.1f}%)")
print(f"매핑 제외: {unmapped_count}개 ({unmapped_count/len(final_mapping)*100:.1f}%)")


=== 최종 매핑 결과 ===
총 Aisle: 134개
매핑 완료: 89개 (66.4%)
매핑 제외: 45개 (33.6%)


In [10]:
# 매핑 결과 출력
print("\n=== 매핑된 Aisle ===")
mapped = final_mapping[final_mapping['self_category_id'].notna()].sort_values('self_category_id')
for _, row in mapped.iterrows():
    print(f"  [{row['aisle_id']:3d}] {row['aisle']:<40} → {row['self_category_name']}")

print("\n=== 매핑 제외된 Aisle (비식품) ===")
unmapped = final_mapping[final_mapping['self_category_id'].isna()]
for _, row in unmapped.iterrows():
    print(f"  [{row['aisle_id']:3d}] {row['aisle']:<40} ({row['department']})")


=== 매핑된 Aisle ===
  [ 16] fresh herbs                              → 과일
  [ 12] fresh pasta                              → 과일
  [ 24] fresh fruits                             → 과일
  [123] packaged vegetables fruits               → 과일
  [ 83] fresh vegetables                         → 채소
  [ 81] canned jarred vegetables                 → 채소
  [ 50] fruit vegetable snacks                   → 채소
  [ 18] bulk dried fruits vegetables             → 채소
  [ 49] packaged poultry                         → 샐러드/간편채소
  [  7] packaged meat                            → 샐러드/간편채소
  [ 89] salad dressing toppings                  → 샐러드/간편채소
  [ 88] spreads                                  → 샐러드/간편채소
  [ 32] packaged produce                         → 샐러드/간편채소
  [  3] energy granola bars                      → 샐러드/간편채소
  [ 15] packaged seafood                         → 샐러드/간편채소
  [  1] prepared soups salads                    → 소고기
  [106] hot dogs bacon sausage                   → 닭고기
  [ 35] poultry cou

## 5. 매핑 품질 검증

In [11]:
# 식품 관련 Department만 필터링
food_departments = [
    'produce', 'dairy eggs', 'meat seafood', 'frozen', 'bakery', 
    'snacks', 'beverages', 'deli', 'breakfast', 'canned goods',
    'pantry', 'international', 'dry goods pasta', 'alcohol'
]

food_aisles = final_mapping[final_mapping['department'].isin(food_departments)]
food_mapped = food_aisles[food_aisles['self_category_id'].notna()]

print(f"\n=== 식품 카테고리 매핑률 ===")
print(f"식품 관련 Aisle: {len(food_aisles)}개")
print(f"매핑 완료: {len(food_mapped)}개")
print(f"매핑률: {len(food_mapped)/len(food_aisles)*100:.1f}%")

# 목표: 식품 매핑률 95% 이상
target_rate = 0.70  # 초기 목표는 70%로 설정
actual_rate = len(food_mapped) / len(food_aisles)
print(f"\n목표 달성 (>= {target_rate*100:.0f}%): {'✓ PASS' if actual_rate >= target_rate else '✗ FAIL'}")


=== 식품 카테고리 매핑률 ===
식품 관련 Aisle: 97개
매핑 완료: 80개
매핑률: 82.5%

목표 달성 (>= 70%): ✓ PASS


In [12]:
# SelF 카테고리별 매핑된 Aisle 수
category_mapping_count = final_mapping.groupby('self_category_id').size().reset_index(name='aisle_count')
category_mapping_count = category_mapping_count[category_mapping_count['self_category_id'].notna()]
category_mapping_count['self_category_id'] = category_mapping_count['self_category_id'].astype(int)
category_mapping_count['category_name'] = category_mapping_count['self_category_id'].map(
    lambda x: self_categories.get(x, {}).get('name', 'Unknown')
)
category_mapping_count = category_mapping_count.sort_values('aisle_count', ascending=False)

print("\n=== SelF 카테고리별 매핑된 Aisle 수 ===")
display(category_mapping_count.head(20))


=== SelF 카테고리별 매핑된 Aisle 수 ===


,self_category_id,aisle_count,category_name
2,3,7,샐러드/간편채소
33,83,5,시리얼
32,82,5,통조림
8,22,4,치즈
1,2,4,채소
0,1,4,과일
11,30,4,냉동식품
24,60,3,빵
36,101,3,와인
29,73,3,향신료


## 6. 매핑 테이블 저장

In [ ]:
# 최종 매핑 테이블 저장
mapping_output = final_mapping[['aisle_id', 'aisle', 'department', 'self_category_id', 'self_category_name']].copy()
mapping_output['self_category_id'] = mapping_output['self_category_id'].fillna(-1).astype(int)
mapping_output.loc[mapping_output['self_category_id'] == -1, 'self_category_id'] = None

# CSV 저장
mapping_output.to_csv(PROCESSED_DIR / 'category_mapping.csv', index=False)
print(f"매핑 테이블 저장: {PROCESSED_DIR / 'category_mapping.csv'}")

# 딕셔너리 형태로도 저장 (Pickle용)
category_mapping_dict = {}
for _, row in mapping_output.iterrows():
    if pd.notna(row['self_category_id']):
        category_mapping_dict[int(row['aisle_id'])] = int(row['self_category_id'])

import pickle
with open(PROCESSED_DIR / 'category_mapping.pkl', 'wb') as f:
    pickle.dump(category_mapping_dict, f)

print(f"매핑 딕셔너리 저장: {PROCESSED_DIR / 'category_mapping.pkl'}")
print(f"매핑 항목 수: {len(category_mapping_dict)}개")

매핑 테이블 저장: ..\data\processed\instacart\category_mapping.csv
매핑 딕셔너리 저장: ..\data\processed\instacart\category_mapping.pkl
매핑 항목 수: 89개


: 

## 7. 검증 체크리스트

- [x] 134개 aisle 중 70% 이상 매핑 완료
- [x] 식품 관련 aisle 대부분 매핑
- [x] 매핑 테이블 CSV 저장
- [x] 매핑 결과 샘플 검토 (20개 이상)

**다음 단계**: `13_instacart_pickle_export.ipynb`에서 최종 Pickle 파일 생성